In [2]:
import scanpy as sc
import pandas as pd
import anndata as ad
import numpy as np

In [50]:
path = "/project/Wellcome_Discovery/datashare/lturiano/data/"

In [51]:
def cap_data(adata, cap=10_000, cell_type_key="cell_type", seed=0):
    rng = np.random.default_rng(seed)

    ct = adata.obs[cell_type_key].to_numpy()
    keep_mask = np.zeros(adata.n_obs, dtype=bool)

    for c in np.unique(ct):
        idx = np.where(ct == c)[0]
        if idx.size <= cap:
            keep_mask[idx] = True
        else:
            keep_idx = rng.choice(idx, size=cap, replace=False)
            keep_mask[keep_idx] = True

    keep_idx = np.sort(np.where(keep_mask)[0])  # preserve original order
    return adata[keep_idx].copy()

In [52]:
def drop_rare_cell_types(
    adata,
    cell_type_key: str = "cell_type",
    min_count: int = 1000,
    copy: bool = True,
    verbose: bool = True,
):
    """
    Drop all cells belonging to cell types that have < min_count cells.

    Returns:
      adata_filt: filtered AnnData
      kept_cell_types: list[str]
      dropped_cell_types: list[tuple[str, int]]  # (cell_type, count)
    """
    if cell_type_key not in adata.obs:
        raise KeyError(f"'{cell_type_key}' not found in adata.obs")

    ct = adata.obs[cell_type_key].astype(str)
    counts = ct.value_counts()

    kept = counts[counts >= min_count].index.tolist()
    dropped = counts[counts < min_count]
    dropped_list = [(k, int(v)) for k, v in dropped.items()]

    mask = ct.isin(kept).to_numpy()
    adata_filt = adata[mask].copy() if copy else adata[mask]

    if verbose:
        print(f"Total cells before: {adata.n_obs:,}")
        print(f"Total cells after : {adata_filt.n_obs:,}")
        print(f"Kept cell types   : {len(kept)}")
        print(f"Dropped cell types: {len(dropped_list)}")
        if dropped_list:
            # show up to first 20
            preview = ", ".join([f"{k}({v})" for k, v in dropped_list[:20]])
            more = "" if len(dropped_list) <= 20 else f", ... (+{len(dropped_list)-20} more)"
            print("Dropped:", preview + more)

    return adata_filt, kept, dropped_list

## Combined single cell and single nuclei RNA-Seq data - Heart Global
Link: https://cellxgene.cziscience.com/collections/3116d060-0a8e-4767-99bb-e866badea1ed  
Dataset code: 9d5eb472-3657-4035-8aea-d3053934e120.h5ad  
Assays: 10x 3' v2, 10x 3' v3, 10x multiome  
Disease: normal

In [70]:
# Load h5ad file
heart = sc.read_h5ad(path + "heart_combined.h5ad")

In [5]:
# select only 10x multiome assay
heart_multiome = heart[heart.obs["assay"] == "10x multiome"]

In [6]:
heart_multiome.obs["cell_type"].value_counts()

cell_type
fibroblast                              63789
regular ventricular cardiac myocyte     47782
endothelial cell                        26288
myeloid cell                            24857
regular atrial cardiac myocyte          19509
mural cell                              14794
lymphocyte                               7403
neural cell                              2680
adipocyte                                2502
mast cell                                 625
endothelial cell of lymphatic vessel      618
mesothelial cell                          213
Name: count, dtype: int64

In [7]:
ct_to_rename = {
    # endothelial
    "endothelial cell of lymphatic vessel": "endothelial cell",

    # myocyte
    "regular atrial cardiac myocyte": "myocyte",
    "regular ventricular cardiac myocyte": "myocyte"
}
# apply the renaming to the cell types in heart_multiome
heart_multiome.obs["cell_type"] = heart_multiome.obs["cell_type"].replace(ct_to_rename)

/tmp/ipykernel_3008479/3286864449.py:10: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  heart_multiome.obs["cell_type"] = heart_multiome.obs["cell_type"].replace(ct_to_rename)
/tmp/ipykernel_3008479/3286864449.py:10: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  heart_multiome.obs["cell_type"] = heart_multiome.obs["cell_type"].replace(ct_to_rename)


In [42]:
heart_multiome.obs["cell_type"].value_counts()

cell_type
myocyte             67291
fibroblast          63789
endothelial cell    26906
myeloid cell        24857
mural cell          14794
lymphocyte           7403
neural cell          2680
adipocyte            2502
mast cell             625
mesothelial cell      213
Name: count, dtype: int64

In [47]:
heart_multiome = cap_data(heart_multiome, cap=10_000, cell_type_key="cell_type", seed=42)

In [48]:
X = heart_multiome.X.toarray()

obs = pd.DataFrame(heart_multiome.obs['cell_type'])
obs["organ"] = "heart"
var = pd.DataFrame(heart_multiome.var['feature_name'])

heart_multiome_filt = ad.AnnData(X=X, obs=obs, var=var)

In [49]:
heart_multiome_filt.write_h5ad(path + "gex/" + "heart_10x_filt.h5ad", compression="gzip")

## Integrated human endoderm-derived organoids cell atlas (HEOCA)
Link: https://cellxgene.cziscience.com/collections/6282a908-f162-44a2-99a3-8a942e4271b2  
Dataset code: c49c8589-0221-4955-b547-270f8d67d13f.h5ad  
Assays: many  
Disease: normal

In [77]:
# Load the .h5ad file
endoderm = sc.read_h5ad(path + "endoderm.h5ad")

In [15]:
endoderm_multiome = endoderm[endoderm.obs["assay"] == "10x multiome"]

In [17]:
ct_to_rename = {
    # epithelial
    "epithelial cell of pancreas": "epithelial cell",

    # duct
    "pancreatic ductal cell": "duct cell",
}

# apply the renaming to the cell types in heart_multiome
endoderm_multiome.obs["cell_type"] = endoderm_multiome.obs["cell_type"].replace(ct_to_rename)

/tmp/ipykernel_3008479/741562607.py:10: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  endoderm_multiome.obs["cell_type"] = endoderm_multiome.obs["cell_type"].replace(ct_to_rename)
/tmp/ipykernel_3008479/741562607.py:10: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  endoderm_multiome.obs["cell_type"] = endoderm_multiome.obs["cell_type"].replace(ct_to_rename)


In [22]:
ct_to_remove = [
    "pancreatic A cell",
    "type B pancreatic cell",
    "pancreatic D cell",
    "pancreatic epsilon cell",
]



endoderm_multiome = endoderm_multiome[~endoderm_multiome.obs["cell_type"].isin(ct_to_remove)]

In [33]:
endoderm_multiome.obs["cell_type"].value_counts()

cell_type
epithelial cell     1588
duct cell            228
acinar cell          103
mesothelial cell       1
mesodermal cell        1
endothelial cell       1
Name: count, dtype: int64

In [39]:
X = endoderm_multiome.X.toarray()

obs = pd.DataFrame(endoderm_multiome.obs['cell_type'])
obs["organ"] = "endoderm"
var = pd.DataFrame(endoderm_multiome.var['feature_name'])

endoderm_multiome_filt = ad.AnnData(X=X, obs=obs, var=var)

In [40]:
endoderm_multiome_filt.write_h5ad(path + "gex/" + "endoderm_10x_filt.h5ad", compression="gzip")

## Single cell atlas of large B-cell lymphoma
Link: https://cellxgene.cziscience.com/collections/4429f25e-5c91-4981-9cf2-6a1044c36732  
Dataset code: 560e2123-d4de-47ad-8f0c-92f8da6c00fe.h5ad  
Assays: 10x multiome  
Disease: normal, B-cell non-Hodgkin lymphoma

In [25]:
# Load h5ad file
lymph_node = sc.read_h5ad(path + "lymph_10x.h5ad")

In [26]:
# take only the disease == normal samples
lymph_node_normal = lymph_node[lymph_node.obs["disease"] == "normal"]

In [27]:
lymph_node_normal.obs["cell_type"].value_counts()

cell_type
CD8-positive, alpha-beta T cell         2909
CD4-positive, alpha-beta T cell         2526
macrophage                              1985
fibroblast                              1570
endothelial cell                        1542
plasmacytoid dendritic cell             1401
regulatory T cell                        819
natural killer cell                      606
T follicular helper cell                 521
monocyte                                 310
pericyte                                 187
follicular dendritic cell                182
dendritic cell                           167
conventional dendritic cell              146
endothelial cell of lymphatic vessel     129
smooth muscle cell                       118
monocyte-derived dendritic cell           55
adipocyte                                 11
Name: count, dtype: int64

In [30]:
ct_to_rename = {
    # T
    "CD8-positive, alpha-beta T cell": "T cell",
    "CD4-positive, alpha-beta T cell": "T cell",
    "regulatory T cell": "T cell",
    "T follicular helper cell": "T cell",
    
    # dendritic
    "plasmacytoid dendritic cell": "dendritic cell",
    "follicular dendritic cell": "dendritic cell",
    "conventional dendritic cell": "dendritic cell",
    "monocyte-derived dendritic cell": "dendritic cell",
    
    # endothelial
    "endothelial cell of lymphatic vessel": "endothelial cell",
}

# apply the renaming to the cell types in heart_multiome
lymph_node_normal.obs["cell_type"] = lymph_node_normal.obs["cell_type"].replace(ct_to_rename)

/tmp/ipykernel_3008479/363844380.py:19: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  lymph_node_normal.obs["cell_type"] = lymph_node_normal.obs["cell_type"].replace(ct_to_rename)
/tmp/ipykernel_3008479/363844380.py:19: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  lymph_node_normal.obs["cell_type"] = lymph_node_normal.obs["cell_type"].replace(ct_to_rename)


In [31]:
lymph_node_normal.obs["cell_type"].value_counts()

cell_type
T cell                 6775
macrophage             1985
dendritic cell         1951
endothelial cell       1671
fibroblast             1570
natural killer cell     606
monocyte                310
pericyte                187
smooth muscle cell      118
adipocyte                11
Name: count, dtype: int64

In [37]:
X = lymph_node_normal.X.toarray()

obs = pd.DataFrame(lymph_node_normal.obs['cell_type'])
obs["organ"] = "lymph"
var = pd.DataFrame(lymph_node_normal.var['feature_name'])

lymph_node_normal_filt = ad.AnnData(X=X, obs=obs, var=var)

In [38]:
lymph_node_normal_filt.write_h5ad(path + "gex/" + "lymph_10x_normal_filt.h5ad", compression="gzip")

## Kidney

In [22]:
kidney = sc.read_h5ad(path + "kidney_10x.h5ad")

In [23]:
kidney.obs["cell_type"].value_counts()

cell_type
kidney proximal convoluted tubule epithelial cell            12856
kidney loop of Henle thick ascending limb epithelial cell     5413
epithelial cell of proximal tubule                            2871
epithelial cell of early distal convoluted tubule             2718
epithelial cell of proximal tubule segment 3                  2583
renal principal cell                                          1984
kidney collecting duct alpha-intercalated cell                1479
kidney connecting tubule epithelial cell                      1331
podocyte                                                       973
kidney connecting tubule principal cell                        973
peritubular capillary endothelial cell                         903
renal beta-intercalated cell                                   726
glomerular capillary endothelial cell                          557
parietal epithelial cell                                       405
fibroblast                                          

In [44]:
ct_to_rename = {
    "plasmacytoid dendritic cell": "dendritic cell",

    # ---- MYELOID ----
    "myeloid dendritic cell": "dendritic cell",
    "non-classical monocyte": "monocyte",
    
    "kidney interstitial alternatively activated macrophage": "macrophage",
    "kidney interstitial macrophage": "macrophage",

    # ---- ENDOTHELIAL ----
    "endothelial cell of arteriole": "endothelial cell",
    "endothelial cell of lymphatic vessel": "endothelial cell",
    "glomerular capillary endothelial cell": "endothelial cell",
    "peritubular capillary endothelial cell": "endothelial cell",
    "vasa recta ascending limb cell": "limb cell",
    "vasa recta descending limb cell": "limb cell",

    # ---- EPITHELIAL (kidney tubules etc.) ----
    "epithelial cell of proximal tubule": "epithelial cell",
    "kidney proximal convoluted tubule epithelial cell": "epithelial cell",
    "kidney loop of Henle thick ascending limb epithelial cell": "epithelial cell",
    "epithelial cell of early distal convoluted tubule": "epithelial cell",
    "epithelial cell of proximal tubule segment": "epithelial cell",
    "epithelial cell of distal tubule segment 3": "epithelial cell",
    "epithelial cell of proximal tubule segment 3": "epithelial cell",
    "epithelial cell of convoluted tubule": "epithelial cell",
    
    "kidney outer medulla collecting duct intercalated cell": "duct cell",
    "kidney collecting duct alpha-intercalated cell": "duct cell",
    "kidney collecting duct intercalated cell": "duct cell",
    "kidney connecting tubule epithelial cell": "epithelial cell",
    "kidney loop of Henle limb epithelial cell": "epithelial cell",
    "kidney proximal tubule epithelial cell": "epithelial cell",
    "macula densa epithelial cell": "epithelial cell",
    "parietal epithelial cell": "epithelial cell",

    # ---- PODOCYTES / MESANGIAL ----
    "podocyte": "podocyte",
    "mesangial cell": "mesangial cell",

    # ---- FIBRO / STROMAL ----
    "fibroblast": "fibroblast",
    "renal interstitial pericyte": "pericyte",
    "vascular associated smooth muscle cell": "smooth muscle cell",

}


# apply the renaming to the cell types in heart_multiome
kidney.obs["cell_type"] = kidney.obs["cell_type"].replace(ct_to_rename)

/tmp/ipykernel_3123019/3534300189.py:51: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  kidney.obs["cell_type"] = kidney.obs["cell_type"].replace(ct_to_rename)
/tmp/ipykernel_3123019/3534300189.py:51: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  kidney.obs["cell_type"] = kidney.obs["cell_type"].replace(ct_to_rename)


In [45]:
ct_to_remove = ["plasma cell", "neutrophil", "kidney connecting tubule principal cell",
                "kidney outer medulla intercalated cell", "renal beta-intercalated cell",
                "renal principal cell"
               ]


kidney = kidney[~kidney.obs["cell_type"].isin(ct_to_remove)]

In [46]:
kidney.obs["cell_type"].value_counts()

cell_type
epithelial cell       28456
endothelial cell       1753
duct cell              1506
podocyte                973
fibroblast              318
mesangial cell          265
T cell                  180
limb cell               153
pericyte                101
smooth muscle cell       91
macrophage               82
dendritic cell           80
B cell                   21
lymphocyte               18
monocyte                 16
Name: count, dtype: int64

In [47]:
kidney = cap_data(kidney, cap=10_000, cell_type_key="cell_type", seed=42)

In [48]:
X = kidney.X.toarray()

obs = pd.DataFrame(kidney.obs['cell_type'])
obs["organ"] = "kidney"
var = pd.DataFrame(kidney.var['feature_name'])

kidney_filt = ad.AnnData(X=X, obs=obs, var=var)

In [49]:
kidney_filt.write_h5ad(path + "gex/" + "kidney_10x_filt.h5ad", compression="gzip")

## Concat multiome data

In [8]:
heart = sc.read_h5ad(path + "gex/heart_10x_filt.h5ad")
endoderm = sc.read_h5ad(path + "gex/endoderm_10x_filt.h5ad")
lymph = sc.read_h5ad(path + "gex/lymph_10x_normal_filt.h5ad")
kidney = sc.read_h5ad(path + "gex/kidney_10x_filt.h5ad")

In [8]:
heart.obs["cell_type"].value_counts()

cell_type
myocyte             10000
fibroblast          10000
endothelial cell    10000
mural cell          10000
myeloid cell        10000
lymphocyte           7403
neural cell          2680
adipocyte            2502
mast cell             625
mesothelial cell      213
Name: count, dtype: int64

In [9]:
endoderm.obs["cell_type"].value_counts()

cell_type
epithelial cell     1588
duct cell            228
acinar cell          103
mesothelial cell       1
mesodermal cell        1
endothelial cell       1
Name: count, dtype: int64

In [10]:
lymph.obs["cell_type"].value_counts()

cell_type
T cell                 6775
macrophage             1985
dendritic cell         1951
endothelial cell       1671
fibroblast             1570
natural killer cell     606
monocyte                310
pericyte                187
smooth muscle cell      118
adipocyte                11
Name: count, dtype: int64

In [9]:
data = [heart, endoderm, lymph, kidney]

# concatenate:
# - join="inner" keeps only common genes across all organs (safest)
# - label adds a batch key (organ); index_unique avoids obs name collisions
data_concat = ad.concat(
    data,          # dict or list
    join="outer",
    axis=0,
    index_unique=None,  # <- keeps original obs_names unchanged
    merge="first"
)

In [10]:
data_concat.shape

(96086, 37989)

In [15]:
data_concat.obs["cell_type"].value_counts()

cell_type
endothelial cell       13425
fibroblast             11888
epithelial cell        11588
myocyte                10000
myeloid cell           10000
mural cell             10000
lymphocyte              7421
T cell                  6955
neural cell             2680
adipocyte               2513
macrophage              2067
dendritic cell          2031
duct cell               1734
podocyte                 973
mast cell                625
natural killer cell      606
monocyte                 326
pericyte                 288
mesangial cell           265
mesothelial cell         214
smooth muscle cell       209
limb cell                153
acinar cell              103
B cell                    21
mesodermal cell            1
Name: count, dtype: int64

In [16]:
data_concat, _, _ = drop_rare_cell_types(data_concat, min_count=1_500)

Total cells before: 96,086
Total cells after : 92,302
Kept cell types   : 13
Dropped cell types: 12
Dropped: podocyte(973), mast cell(625), natural killer cell(606), monocyte(326), pericyte(288), mesangial cell(265), mesothelial cell(214), smooth muscle cell(209), limb cell(153), acinar cell(103), B cell(21), mesodermal cell(1)


In [17]:
data_concat.obs["cell_type"].value_counts()

cell_type
endothelial cell    13425
fibroblast          11888
epithelial cell     11588
mural cell          10000
myeloid cell        10000
myocyte             10000
lymphocyte           7421
T cell               6955
neural cell          2680
adipocyte            2513
macrophage           2067
dendritic cell       2031
duct cell            1734
Name: count, dtype: int64

In [18]:
data_concat.write_h5ad(path + "gex_filt.h5ad", compression="gzip")

# scRNA data

## Heart (same as before)

In [71]:
# select only rna samples
heart_rna = heart[heart.obs["suspension_type"] == "cell"]

In [72]:
ct_to_rename = {
    # endothelial
    "endothelial cell of lymphatic vessel": "endothelial cell",

    # myocyte
    "regular atrial cardiac myocyte": "myocyte",
    "regular ventricular cardiac myocyte": "myocyte"
}
# apply the renaming to the cell types in heart_multiome
heart_rna.obs["cell_type"] = heart_rna.obs["cell_type"].replace(ct_to_rename)

/tmp/ipykernel_3123019/2812152133.py:10: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  heart_rna.obs["cell_type"] = heart_rna.obs["cell_type"].replace(ct_to_rename)
/tmp/ipykernel_3123019/2812152133.py:10: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  heart_rna.obs["cell_type"] = heart_rna.obs["cell_type"].replace(ct_to_rename)


In [73]:
heart_rna.obs["cell_type"].value_counts()

cell_type
endothelial cell    80629
mural cell          22641
myeloid cell        14702
lymphocyte          11945
fibroblast           3163
myocyte              1924
neural cell           497
mesothelial cell      195
adipocyte               1
mast cell               1
Name: count, dtype: int64

In [74]:
heart_rna = cap_data(heart_rna, cap=10_000, cell_type_key="cell_type", seed=42)

In [75]:
X = heart_rna.X.toarray()

obs = pd.DataFrame(heart_rna.obs['cell_type'])
obs["organ"] = "heart"
var = pd.DataFrame(heart_rna.var['feature_name'])

heart_rna_filt = ad.AnnData(X=X, obs=obs, var=var)

In [76]:
heart_rna_filt.write_h5ad(path + "rna/" + "heart_rna_filt.h5ad", compression="gzip")

## Endoderm (same as before)

In [78]:
endoderm_rna = endoderm[endoderm.obs["suspension_type"] == "cell"]

In [79]:
ct_to_rename = {
    # muscle
    "smooth muscle cell": "muscle cell",

    # enterocyte
    "enterocyte": "enterocyte",
    "BEST4+ enterocyte": "enterocyte",

    # epithelial
    "epithelial cell of pancreas": "epithelial cell",
    "colon epithelial cell": "epithelial cell",
    "epithelial cell of stomach": "epithelial cell",

    # duct
    "pancreatic ductal cell": "duct cell",

    # alveolar
    "pulmonary alveolar type 1 cell": "alveolar cell",
    "pulmonary alveolar type 2 cell": "alveolar cell"
}

# apply the renaming to the cell types in endoderm_rna
endoderm_rna.obs["cell_type"] = endoderm_rna.obs["cell_type"].replace(ct_to_rename)

/tmp/ipykernel_3123019/2400893257.py:23: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  endoderm_rna.obs["cell_type"] = endoderm_rna.obs["cell_type"].replace(ct_to_rename)
/tmp/ipykernel_3123019/2400893257.py:23: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  endoderm_rna.obs["cell_type"] = endoderm_rna.obs["cell_type"].replace(ct_to_rename)


In [80]:
ct_to_remove = [
    "enteroendocrine cell",
    "hematopoietic cell",
    "hepatoblast",
    "interstitial cell of Cajal",
    "intestinal tuft cell",
    "Kupffer cell",
    "luminal cell of prostate epithelium",
    "lung secretory cell",
    "M cell of gut",
    "mesodermal cell",
    "neuroendocrine cell",
    "neuron",
    "pancreatic A cell",
    "pancreatic B cell",
    "pancreatic D cell",
    "pancreatic epsilon cell",
    "paneth cell",
    "pericyte",
    "Schwann cell",
    "stem cell",
    "thymocyte",
    "type B pancreatic cell",
]

# apply the removal of cell types to endoderm_rna
endoderm_rna = endoderm_rna[~endoderm_rna.obs["cell_type"].isin(ct_to_remove)]

In [81]:
endoderm_rna.obs["cell_type"].value_counts()

cell_type
epithelial cell          99653
alveolar cell            77310
basal cell               45155
goblet cell              44877
enterocyte               31303
club cell                30960
stromal cell             27265
hepatic stellate cell    26220
acinar cell              10432
muscle cell               9839
duct cell                 7295
ciliated cell             6846
mesothelial cell          3341
endothelial cell           932
Name: count, dtype: int64

In [82]:
endoderm_rna = cap_data(endoderm_rna, cap=10_000, cell_type_key="cell_type", seed=42)

In [83]:
X = endoderm_rna.X.toarray()

obs = pd.DataFrame(endoderm_rna.obs['cell_type'])
obs["organ"] = "endoderm"
var = pd.DataFrame(endoderm_rna.var['feature_name'])

endoderm_rna_filt = ad.AnnData(X=X, obs=obs, var=var)

In [84]:
endoderm_rna_filt.write_h5ad(path + "rna/" + "endoderm_rna_filt.h5ad", compression="gzip")

## Kidney

In [5]:
kidney_rna = sc.read_h5ad(path + "kidney_rna.h5ad")

In [6]:
kidney_rna.obs["cell_type"].value_counts()

cell_type
epithelial cell of proximal tubule                                            78480
kidney loop of Henle thick ascending limb epithelial cell                     70595
kidney collecting duct principal cell                                         33965
endothelial cell                                                              27510
naive thymus-derived CD4-positive, alpha-beta T cell                          15268
kidney connecting tubule epithelial cell                                      15236
kidney collecting duct intercalated cell                                      14057
kidney distal convoluted tubule epithelial cell                               13146
kidney loop of Henle thin descending limb epithelial cell                     11464
renal interstitial pericyte                                                    7960
effector memory CD8-positive, alpha-beta T cell                                6617
kidney loop of Henle thin ascending limb epithelial cell          

In [26]:
ct_to_rename = {


    # Dendritic
    "myeloid dendritic cell": "dendritic cell",
    "migratory dendtritic cell": "dendritic cell",
    "plasmacytoid dendritic cell": "dendritic cell",
    "CD8-alpha-positive thymic conventional dendritic cell": "dendritic cell",
    "CD8-alpha-negative thymic conventional dendritic cell": "dendritic cell",
    "plasmacytoid dendritic cell, human": "dendritic cell",
    
    # Monocyte
    "non-classical monocyte": "monocyte",
    "classical monocyte": "monocyte",
    
    # Macrophage
    "kidney resident macrophage": "macrophage",
    "kidney interstitial alternatively activated macrophage": "macrophage",
    "kidney interstitial macrophage": "macrophage",
    "cycling macrophage": "macrophage",

    # ---- ENDOTHELIAL ----
    "endothelial cell of arteriole": "endothelial cell",
    "endothelial cell of lymphatic vessel": "endothelial cell",
    "glomerular capillary endothelial cell": "endothelial cell",
    "peritubular capillary endothelial cell": "endothelial cell",
    "vasa recta ascending limb cell": "limb cell",
    "vasa recta descending limb cell": "limb cell",

    # ---- EPITHELIAL (kidney tubules etc.) ----
    "epithelial cell of proximal tubule": "epithelial cell",
    "kidney proximal convoluted tubule epithelial cell": "epithelial cell",
    "kidney loop of Henle thick ascending limb epithelial cell": "epithelial cell",
    "epithelial cell of early distal convoluted tubule": "epithelial cell",
    "epithelial cell of proximal tubule segment": "epithelial cell",
    "epithelial cell of distal tubule segment 3": "epithelial cell",
    "epithelial cell of proximal tubule segment 3": "epithelial cell",
    "epithelial cell of convoluted tubule": "epithelial cell",
    "kidney distal convoluted tubule epithelial cell": "epithelial cell",
    
    "kidney outer medulla collecting duct intercalated cell": "duct cell",
    "kidney collecting duct alpha-intercalated cell": "duct cell",
    "kidney collecting duct intercalated cell": "duct cell",
    "kidney collecting duct principal cell": "duct cell",
    "kidney connecting tubule epithelial cell": "epithelial cell",
    "kidney loop of Henle limb epithelial cell": "epithelial cell",
    "kidney proximal tubule epithelial cell": "epithelial cell",
    "macula densa epithelial cell": "epithelial cell",
    "parietal epithelial cell": "epithelial cell",
    "kidney loop of Henle thin descending limb epithelial cell": "epithelial cell",
    "kidney loop of Henle thin ascending limb epithelial cell": "eptihelial cell",

    # ---- PODOCYTES / MESANGIAL ----
    "podocyte": "podocyte",
    "mesangial cell": "mesangial cell",

    # Fibroblast
    "kidney interstitial fibroblast": "fibroblast",
    
    "renal interstitial pericyte": "pericyte",
    "vascular associated smooth muscle cell": "smooth muscle cell",
    
    # T
    "naive thymus-derived CD4-positive, alpha-beta T cell": "T cell",
    "effector memory CD8-positive, alpha-beta T cell": "T cell",
    "effector memory CD8-positive, alpha-beta T cell, terminally differentiated": "T cell",
    "naive regulatory T cell": "T cell",
    "mucosal invariant T cell": "T cell",
    
    "group 3 innate lymphoid cell": "lymphoid",

}

# apply the renaming
kidney_rna.obs["cell_type"] = kidney_rna.obs["cell_type"].replace(ct_to_rename)

/tmp/ipykernel_2563990/1157231394.py:76: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  kidney_rna.obs["cell_type"] = kidney_rna.obs["cell_type"].replace(ct_to_rename)
/tmp/ipykernel_2563990/1157231394.py:76: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  kidney_rna.obs["cell_type"] = kidney_rna.obs["cell_type"].replace(ct_to_rename)


In [27]:
ct_to_remove = ["plasma cell", "neutrophil", "kidney connecting tubule principal cell",
                "kidney outer medulla intercalated cell", "renal beta-intercalated cell",
                "renal principal cell"
               ]


kidney_rna = kidney_rna[~kidney_rna.obs["cell_type"].isin(ct_to_remove)]

In [30]:
kidney_rna.obs["cell_type"].value_counts()

cell_type
endothelial cell       10000
T cell                 10000
duct cell              10000
epithelial cell        10000
pericyte                7960
macrophage              7620
monocyte                6794
eptihelial cell         6245
B cell                  5609
dendritic cell          4522
natural killer cell     3603
neural cell             2621
fibroblast              1729
podocyte                 840
lymphoid                 355
mast cell                195
Name: count, dtype: int64

In [29]:
kidney_rna = cap_data(kidney_rna, cap=10_000, cell_type_key="cell_type", seed=42)

In [31]:
X = kidney_rna.X.toarray()

obs = pd.DataFrame(kidney_rna.obs['cell_type'])
obs["organ"] = "kidney"
var = pd.DataFrame(kidney_rna.var['feature_name'])

kidney_rna_filt = ad.AnnData(X=X, obs=obs, var=var)

In [32]:
kidney_rna_filt.write_h5ad(path + "rna/" + "kidney_rna_filt.h5ad", compression="gzip")

## Lymph node
Link: https://cellxgene.cziscience.com/collections/854c0855-23ad-4362-8b77-6b1639e7a9fc

In [33]:
lymph_rna = sc.read_h5ad(path + "lymph_rna.h5ad")

In [34]:
lymph_rna.obs["cell_type"].value_counts()

cell_type
central memory CD4-positive, alpha-beta T cell           37025
memory B cell                                            17454
regulatory T cell                                        13596
T follicular helper cell                                 11986
activated CD4-positive, alpha-beta T cell                 9383
naive B cell                                              8718
effector memory CD8-positive, alpha-beta T cell           7552
central memory CD8-positive, alpha-beta T cell            4795
effector memory CD4-positive, alpha-beta T cell           3368
CD16-negative, CD56-bright natural killer cell, human     1647
T cell                                                    1545
innate lymphoid cell                                      1154
B cell                                                    1070
plasma cell                                               1041
macrophage                                                 581
plasmablast                                  

In [35]:
ct_to_rename = {

    # ---- T CELLS ----
    "activated CD4-positive, alpha-beta T cell": "T cell",
    "central memory CD4-positive, alpha-beta T cell": "T cell",
    "central memory CD8-positive, alpha-beta T cell": "T cell",
    "effector memory CD4-positive, alpha-beta T cell": "T cell",
    "effector memory CD8-positive, alpha-beta T cell": "T cell",
    "memory B cell": "B cell",   # moved below but kept for clarity
    "mucosal invariant T cell": "T cell",
    "regulatory T cell": "T cell",
    "T follicular helper cell": "T cell",
    "T cell": "T cell",

    # ---- B CELLS ----
    "B cell": "B cell",
    "naive B cell": "B cell",
    "memory B cell": "B cell",
    "germinal center B cell": "B cell",

    # ---- PLASMA ----
    "plasma cell": "plasma cell",
    "plasmablast": "plasma cell",

    # ---- NK / INNATE LYMPHOID ----
    "CD16-negative, CD56-bright natural killer cell, human": "natural killer cell",
    "CD16-positive, CD56-dim natural killer cell, human": "natural killer cell",
    "innate lymphoid cell": "lymphoid",

    # ---- MONOCYTES / MACROPHAGES ----
    "classical monocyte": "monocyte",
    "non-classical monocyte": "monocyte",
    "macrophage": "macrophage",

    # ---- DENDRITIC CELLS ----
    "conventional dendritic cell": "dendritic cell",
    "plasmacytoid dendritic cell": "dendritic cell",
    "dendritic cell, human": "dendritic cell",

    # ---- ENDOTHELIAL ----
    "endothelial cell": "endothelial cell",
    "endothelial cell of lymphatic vessel": "endothelial cell",

    # ---- STROMAL ----
    "stromal cell": "stromal cell",

    # ---- MAST ----
    "mast cell": "mast cell",
}

# apply the renaming
lymph_rna.obs["cell_type"] = lymph_rna.obs["cell_type"].replace(ct_to_rename)

/tmp/ipykernel_2563990/196372598.py:52: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  lymph_rna.obs["cell_type"] = lymph_rna.obs["cell_type"].replace(ct_to_rename)


In [40]:
lymph_rna.obs["cell_type"].value_counts()

cell_type
B cell                 10000
T cell                 10000
natural killer cell     2174
plasma cell             1621
lymphoid                1154
endothelial cell         789
macrophage               581
dendritic cell           467
stromal cell             225
monocyte                 195
mast cell                 81
Name: count, dtype: int64

In [37]:
lymph_rna = cap_data(lymph_rna, cap=10_000, cell_type_key="cell_type", seed=42)

In [38]:
X = lymph_rna.X.toarray()

obs = pd.DataFrame(lymph_rna.obs['cell_type'])
obs["organ"] = "lymph"
var = pd.DataFrame(lymph_rna.var['feature_name'])

lymph_rna_filt = ad.AnnData(X=X, obs=obs, var=var)

In [39]:
lymph_rna_filt.write_h5ad(path + "rna/" + "lymph_rna_filt.h5ad", compression="gzip")

## Concat sn data (other .py file)

## Concat sc data (other .py file)

## Align input (multiome) and output (sc) datasets

In [11]:
rna = sc.read_h5ad(path + "rna_filt.h5ad")
gex = sc.read_h5ad(path + "gex_filt.h5ad")

In [21]:
rna.shape, gex.shape

((258671, 54206), (92302, 37989))

In [22]:
rna.obs["cell_type"].value_counts()

cell_type
B cell                   15000
T cell                   15000
duct cell                15000
epithelial cell          15000
endothelial cell         15000
stromal cell             10225
enterocyte               10000
basal cell               10000
alveolar cell            10000
club cell                10000
hepatic stellate cell    10000
lymphocyte               10000
goblet cell              10000
acinar cell              10000
myeloid cell             10000
mural cell               10000
muscle cell               9839
macrophage                8201
pericyte                  7960
monocyte                  6989
ciliated cell             6846
eptihelial cell           6245
natural killer cell       5777
dendritic cell            4989
fibroblast                4892
mesothelial cell          3536
neural cell               3118
myocyte                   1924
plasma cell               1621
lymphoid                  1509
Name: count, dtype: int64

In [23]:
gex.obs["cell_type"].value_counts()

cell_type
endothelial cell    13425
fibroblast          11888
epithelial cell     11588
mural cell          10000
myeloid cell        10000
myocyte             10000
lymphocyte           7421
T cell               6955
neural cell          2680
adipocyte            2513
macrophage           2067
dendritic cell       2031
duct cell            1734
Name: count, dtype: int64

In [24]:
rna.X.max(), gex.X.max()

(np.float32(8.969073), np.float64(10.076958656311035))

In [25]:
rna.X.min(), gex.X.min()

(np.float32(0.0), np.float64(0.0))

In [13]:
import numpy as np
import scanpy as sc
import anndata as ad

def align_anndata_by_cell_type(
    rna: ad.AnnData,
    gex: ad.AnnData,
    cell_type_key: str = "cell_type",
    seed: int = 0,
    copy: bool = True,
):
    """
    Align two AnnData objects by cell type so that:
      1) They contain only the intersection of cell types.
      2) For each shared cell type, both are downsampled to the same count
         (the min count across the two datasets for that cell type).
      3) The resulting AnnData objects have the same number of cells and
         are ordered so that for every index i, rna_aligned.obs[cell_type_key][i]
         == gex_aligned.obs[cell_type_key][i].

    Returns
    -------
    rna_aligned, gex_aligned, info_dict
    """
    if cell_type_key not in rna.obs or cell_type_key not in gex.obs:
        raise KeyError(f"'{cell_type_key}' must be present in both .obs")

    # Work on views or copies
    rna0 = rna.copy() if copy else rna
    gex0 = gex.copy() if copy else gex

    # Ensure strings (avoid categorical mismatch headaches)
    rna0.obs[cell_type_key] = rna0.obs[cell_type_key].astype(str)
    gex0.obs[cell_type_key] = gex0.obs[cell_type_key].astype(str)

    # Shared cell types
    ct_rna = set(rna0.obs[cell_type_key].unique())
    ct_gex = set(gex0.obs[cell_type_key].unique())
    shared_ct = sorted(ct_rna.intersection(ct_gex))

    if len(shared_ct) == 0:
        raise ValueError("No shared cell types between the two AnnData objects.")

    # Subset to shared cell types only
    rna0 = rna0[rna0.obs[cell_type_key].isin(shared_ct)].copy()
    gex0 = gex0[gex0.obs[cell_type_key].isin(shared_ct)].copy()

    # Counts per cell type
    rna_counts = rna0.obs[cell_type_key].value_counts()
    gex_counts = gex0.obs[cell_type_key].value_counts()

    # Determine per-cell-type target (min across datasets)
    target = {ct: int(min(rna_counts.get(ct, 0), gex_counts.get(ct, 0))) for ct in shared_ct}
    # Drop any ct that ends up with 0 (paranoia / safety)
    target = {ct: n for ct, n in target.items() if n > 0}
    shared_ct = sorted(target.keys())

    if len(shared_ct) == 0:
        raise ValueError("After computing min counts, no cell types have >0 cells in both datasets.")

    rng = np.random.default_rng(seed)

    def _sample_indices(adata_obj: ad.AnnData, ct: str, n: int):
        idx = np.where(adata_obj.obs[cell_type_key].values == ct)[0]
        if len(idx) < n:
            raise ValueError(f"Not enough cells for cell type '{ct}' (needed {n}, found {len(idx)}).")
        return rng.choice(idx, size=n, replace=False)

    # Sample indices per CT for each dataset
    rna_sel = np.concatenate([_sample_indices(rna0, ct, target[ct]) for ct in shared_ct])
    gex_sel = np.concatenate([_sample_indices(gex0, ct, target[ct]) for ct in shared_ct])

    # Reorder within each CT so alignment is consistent/deterministic:
    # we build a final "ct-ordered" list where all cells of ct1 come first, then ct2, ...
    # and within each CT we keep a stable order by sorting by the selected indices.
    def _ct_ordered(adata_obj: ad.AnnData, selected_idx: np.ndarray):
        adata_sub = adata_obj[selected_idx].copy()
        # stable sort by (cell_type, original selection index order)
        # We'll sort by cell_type then by obs_names to keep it deterministic.
        order = np.lexsort((adata_sub.obs_names.values, adata_sub.obs[cell_type_key].values))
        return adata_sub[order].copy()

    rna_aligned = _ct_ordered(rna0, rna_sel)
    gex_aligned = _ct_ordered(gex0, gex_sel)

    # Final sanity check: aligned cell types at every position
    if not np.array_equal(
        rna_aligned.obs[cell_type_key].values,
        gex_aligned.obs[cell_type_key].values
    ):
        raise RuntimeError("Alignment failed: cell types are not identical row-by-row after ordering.")

    info = {
        "shared_cell_types": shared_ct,
        "target_per_cell_type": target,
        "n_cells_final": int(rna_aligned.n_obs),
    }
    return rna_aligned, gex_aligned, info

In [ ]:
rna_aligned, gex_aligned, info = align_anndata_by_cell_type(rna, gex, cell_type_key="cell_type", seed=42)
print(info)
assert (rna_aligned.obs["cell_type"].values == gex_aligned.obs["cell_type"].values).all()

In [ ]:
rna_aligned.obs["cell_type"].value_counts()

In [ ]:
gex_aligned.obs["cell_type"].value_counts()

In [ ]:
rna_aligned.write_h5ad(path + "rna_filt_aligned.h5ad", compression="gzip")
gex_aligned.write_h5ad(path + "gex_filt_aligned.h5ad", compression="gzip")

## Analyse final data

In [56]:
rna_aligned_filt = sc.read_h5ad(path + "rna_filt_aligned.h5ad")
gex_aligned_filt = sc.read_h5ad(path + "gex_filt_aligned.h5ad")

In [25]:
rna_aligned_filt.obs["cell_type"].value_counts()

cell_type
endothelial cell    13425
epithelial cell     11588
mural cell          10000
myeloid cell        10000
lymphocyte           7421
T cell               6955
fibroblast           4892
neural cell          2680
macrophage           2067
dendritic cell       2031
myocyte              1924
duct cell            1734
Name: count, dtype: int64

In [26]:
gex_aligned_filt.obs["cell_type"].value_counts()

cell_type
endothelial cell    13425
epithelial cell     11588
mural cell          10000
myeloid cell        10000
lymphocyte           7421
T cell               6955
fibroblast           4892
neural cell          2680
macrophage           2067
dendritic cell       2031
myocyte              1924
duct cell            1734
Name: count, dtype: int64

In [27]:
rna_aligned_filt.shape, gex_aligned_filt.shape

((74717, 54206), (74717, 37989))

In [57]:
import scipy.sparse as sp

def x_stats(adata, name):
    X = adata.X
    if sp.issparse(X):
        data = X.data
        xmin = data.min() if data.size else 0.0
        xmax = data.max() if data.size else 0.0
        xmean = data.mean() if data.size else 0.0
    else:
        xmin, xmax, xmean = X.min(), X.max(), X.mean()

    print(f"\n{name}")
    print("shape:", adata.shape)
    print("dtype:", X.dtype)
    print("min/max/mean:", float(xmin), float(xmax), float(xmean))

x_stats(rna_aligned_filt, "RNA")
x_stats(gex_aligned_filt, "GEX")


RNA
shape: (74717, 54206)
dtype: float32
min/max/mean: 0.0 8.969073295593262 0.04242892563343048

GEX
shape: (74717, 37989)
dtype: float32
min/max/mean: 0.0 10.076958656311035 0.08460833877325058


In [14]:
# Check if rna has duplicate genes
gene_names = list(rna_aligned_filt.var_names)
print(len(gene_names))
set(gene_names)
print(len(gene_names))

54206
54206


In [15]:
# Check if gex has duplicate genes
gene_names = list(gex_aligned_filt.var_names)
print(len(gene_names))
set(gene_names)
print(len(gene_names))

37989
37989


In [31]:
import numpy as np
from scipy import sparse

def row_sums(X):
    return np.asarray(X.sum(axis=1)).ravel() if sparse.issparse(X) else X.sum(axis=1)

rna_s = row_sums(rna_aligned_filt.X)
gex_s = row_sums(gex_aligned_filt.X)

print("RNA sum/cell percentiles:", np.percentile(rna_s, [0, 1, 50, 99, 100]))
print("GEX sum/cell percentiles:", np.percentile(gex_s, [0, 1, 50, 99, 100]))

RNA sum/cell percentiles: [ 210.44325256 1040.77426758 2185.38061523 4364.51738281 5154.55273438]
GEX sum/cell percentiles: [  615.91894531  1269.35645508  2831.43530273 10502.56796875
 15133.02246094]


In [33]:
def expm1_row_sums(X):
    if sparse.issparse(X):
        # safe: expm1 only on stored data
        Y = X.copy()
        Y.data = np.expm1(Y.data)
        return np.asarray(Y.sum(axis=1)).ravel()
    else:
        return np.expm1(X).sum(axis=1)

rna_counts_like = expm1_row_sums(rna_aligned_filt.X)
gex_counts_like = expm1_row_sums(gex_aligned_filt.X)

print("RNA expm1-sum/cell median:", np.median(rna_counts_like))
print("GEX expm1-sum/cell median:", np.median(gex_counts_like))

RNA expm1-sum/cell median: 9992.25
GEX expm1-sum/cell median: 9973.298


In [58]:
sc.pp.filter_genes(rna_aligned_filt, min_cells=rna_aligned_filt.n_obs*0.30)
sc.pp.filter_genes(gex_aligned_filt, min_cells=gex_aligned_filt.n_obs*0.30)

In [59]:
a = set(list(rna_aligned_filt.var["feature_name"].value_counts().index))
b = set(list(gex_aligned_filt.var["feature_name"].value_counts().index))

In [60]:
len(a.difference(b))

579

In [61]:
len(b.difference(a))

962

# Final check to see if input-output datasets are paired in the correct way

In [34]:
import numpy as np
from scipy import sparse

def _as_1d(a):
    return np.asarray(a).ravel()

def _count_nan_inf(X):
    if sparse.issparse(X):
        d = X.data
        return int(np.isnan(d).sum()), int(np.isinf(d).sum())
    else:
        return int(np.isnan(X).sum()), int(np.isinf(X).sum())

def _min_max_mean(X, n_sample=200_000, seed=0):
    rng = np.random.default_rng(seed)
    if sparse.issparse(X):
        d = X.data
        if d.size == 0:
            return 0.0, 0.0, 0.0
        take = min(n_sample, d.size)
        samp = d[rng.choice(d.size, size=take, replace=False)] if d.size > take else d
        return float(np.min(samp)), float(np.max(samp)), float(np.mean(samp))
    else:
        arr = X
        flat = arr.ravel()
        take = min(n_sample, flat.size)
        samp = flat[rng.choice(flat.size, size=take, replace=False)] if flat.size > take else flat
        return float(np.min(samp)), float(np.max(samp)), float(np.mean(samp))

def _row_sums(X):
    return _as_1d(X.sum(axis=1)) if sparse.issparse(X) else X.sum(axis=1)

def _expm1_row_sums(X):
    # interpret X as log1p normalized; convert back and sum
    if sparse.issparse(X):
        Y = X.copy()
        Y.data = np.expm1(Y.data)
        return _as_1d(Y.sum(axis=1))
    else:
        return np.expm1(X).sum(axis=1)

def _ensure_float32_X(adata):
    X = adata.X
    if sparse.issparse(X):
        if X.dtype != np.float32:
            adata.X = X.astype(np.float32)
    else:
        if adata.X.dtype != np.float32:
            adata.X = adata.X.astype(np.float32)

def preflight_for_dual_loader(
    adata1,
    adata2,
    cell_type_key="cell_type",
    require_same_obs_names=True,
    require_same_cell_type_per_row=True,
    fix_float32=True,
    check_log1p_target_sum=True,
    target_sum=1e4,
    verbose=True,
):
    report = {"ok": True, "issues": [], "warnings": []}

    def issue(msg):
        report["ok"] = False
        report["issues"].append(msg)

    def warn(msg):
        report["warnings"].append(msg)

    # ---------- basic required checks ----------
    if adata1.n_obs != adata2.n_obs:
        issue(f"n_obs mismatch: adata1 {adata1.n_obs} vs adata2 {adata2.n_obs} (loader will break).")
        return report  # cannot proceed safely

    if cell_type_key not in adata1.obs:
        issue(f"'{cell_type_key}' missing in adata1.obs (sampler needs it).")
    if cell_type_key not in adata2.obs:
        warn(f"'{cell_type_key}' missing in adata2.obs (not required by loader, but alignment checks use it).")

    # ---------- obs_names alignment ----------
    if require_same_obs_names:
        if not np.array_equal(adata1.obs_names.to_numpy(), adata2.obs_names.to_numpy()):
            issue("obs_names are NOT identical in the same order (row-wise pairing will be wrong).")
    else:
        warn("Skipping strict obs_names check (you may be pairing wrong cells if rows aren’t aligned).")

    # ---------- cell_type per-row alignment ----------
    if (cell_type_key in adata1.obs) and (cell_type_key in adata2.obs) and require_same_cell_type_per_row:
        ct1 = adata1.obs[cell_type_key].astype(str).to_numpy()
        ct2 = adata2.obs[cell_type_key].astype(str).to_numpy()
        if not np.array_equal(ct1, ct2):
            # show how many differ
            n_diff = int(np.sum(ct1 != ct2))
            issue(f"cell_type mismatch per-row: {n_diff}/{adata1.n_obs} rows differ (pairing likely wrong).")
    elif require_same_cell_type_per_row:
        warn("Could not check per-row cell_type alignment (cell_type_key missing in one obs).")

    # ---------- gene metadata sanity (not required by loader, but good) ----------
    # duplicates can create headaches later (alignment, intersection, etc.)
    if adata1.var_names.has_duplicates:
        warn("adata1.var_names has duplicates (consider making unique).")
    if adata2.var_names.has_duplicates:
        warn("adata2.var_names has duplicates (consider making unique).")

    # ---------- X checks ----------
    for name, adata in [("adata1", adata1), ("adata2", adata2)]:
        X = adata.X
        if X is None:
            issue(f"{name}.X is None.")
            continue

        n_nan, n_inf = _count_nan_inf(X)
        if n_nan > 0:
            issue(f"{name}.X contains NaNs: {n_nan} (replace them before training).")
        if n_inf > 0:
            issue(f"{name}.X contains inf/-inf: {n_inf} (replace them before training).")

        # basic distribution (sampled)
        mn, mx, mean = _min_max_mean(X)
        if verbose:
            print(f"\n{name} X summary")
            print(f"  shape: {adata.shape}")
            print(f"  type : {'sparse' if sparse.issparse(X) else 'dense'}")
            print(f"  dtype: {X.dtype}")
            print(f"  sampled min/max/mean: {mn:.4g} / {mx:.4g} / {mean:.4g}")

        # negative values are unusual for log1p normalized counts
        if mn < -1e-6:
            warn(f"{name}.X has negative values (min~{mn:.3g}). If this is scaled data, confirm model expects it.")

    # ---------- cell type distribution (important for sampler quality) ----------
    if cell_type_key in adata1.obs:
        ct = adata1.obs[cell_type_key].astype(str)
        counts = ct.value_counts()
        if verbose:
            print("\nCell type counts (adata1):")
            print(counts.head(20))
            if len(counts) > 20:
                print(f"... (+{len(counts)-20} more)")

        # sampler creates batches per cell type; tiny groups can lead to lots of tiny last batches
        too_small = counts[counts < 2].index.tolist()
        if len(too_small) > 0:
            warn(f"{len(too_small)} cell types have <2 cells (sampler/batching may be weird).")

    # ---------- normalization scale (optional) ----------
    if check_log1p_target_sum:
        try:
            s1 = _expm1_row_sums(adata1.X)
            s2 = _expm1_row_sums(adata2.X)
            med1, med2 = float(np.median(s1)), float(np.median(s2))
            if verbose:
                print("\nUndo-log check (median expm1 sum per cell):")
                print(f"  adata1: {med1:.2f}")
                print(f"  adata2: {med2:.2f}")

            # relative tolerance: allow 10% drift
            if not (abs(med1 - target_sum) / target_sum < 0.1):
                warn(f"adata1 median expm1-sum {med1:.1f} not near target {target_sum:.0f} (is it log1p-normalized?).")
            if not (abs(med2 - target_sum) / target_sum < 0.1):
                warn(f"adata2 median expm1-sum {med2:.1f} not near target {target_sum:.0f} (is it log1p-normalized?).")

            # check both are close to each other
            if abs(med1 - med2) / max(med1, med2) > 0.1:
                warn(f"adata1/adata2 expm1-sum medians differ >10% ({med1:.1f} vs {med2:.1f}).")
        except Exception as e:
            warn(f"Normalization scale check failed (maybe not log1p?): {repr(e)}")

    # ---------- dtype fix (recommended) ----------
    if fix_float32:
        _ensure_float32_X(adata1)
        _ensure_float32_X(adata2)
        if verbose:
            print("\nConverted X dtypes to float32 where needed.")

    # ---------- final verdict ----------
    if verbose:
        print("\n=== PRE-FLIGHT RESULT ===")
        print("OK?" , report["ok"])
        if report["issues"]:
            print("Issues:")
            for m in report["issues"]:
                print(" -", m)
        if report["warnings"]:
            print("Warnings:")
            for m in report["warnings"]:
                print(" -", m)

    return report

In [41]:
# Overwrite obs_names
rna_aligned_filt.obs_names = rna_aligned_filt.obs_names.astype(str)
rna_aligned_filt.obs_names = [str(i) for i in range(rna_aligned_filt.n_obs)]

# Overwrite obs_names
gex_aligned_filt.obs_names = gex_aligned_filt.obs_names.astype(str)
gex_aligned_filt.obs_names = [str(i) for i in range(gex_aligned_filt.n_obs)]

In [42]:
report = preflight_for_dual_loader(rna_aligned_filt, gex_aligned_filt, cell_type_key="cell_type")


adata1 X summary
  shape: (74717, 54206)
  type : dense
  dtype: float32
  sampled min/max/mean: 0 / 6.181 / 0.04299

adata2 X summary
  shape: (74717, 37989)
  type : dense
  dtype: float32
  sampled min/max/mean: 0 / 8.498 / 0.0845

Cell type counts (adata1):
cell_type
endothelial cell    13425
epithelial cell     11588
mural cell          10000
myeloid cell        10000
lymphocyte           7421
T cell               6955
fibroblast           4892
neural cell          2680
macrophage           2067
dendritic cell       2031
myocyte              1924
duct cell            1734
Name: count, dtype: int64

Undo-log check (median expm1 sum per cell):
  adata1: 9992.25
  adata2: 9973.30

Converted X dtypes to float32 where needed.

=== PRE-FLIGHT RESULT ===
OK? True


In [46]:
import numpy as np

assert gex_aligned_filt.n_obs == rna_aligned_filt.n_obs

ct_gex = gex_aligned_filt.obs["cell_type"].astype(str).to_numpy()
ct_rna = rna_aligned_filt.obs["cell_type"].astype(str).to_numpy()

same = (ct_gex == ct_rna)
print("Row-wise same cell_type:", same.mean(), "=", same.sum(), "/", same.size)

# If not perfect, show where it breaks
if not np.all(same):
    bad = np.where(~same)[0][:20]
    print("First mismatches at indices:", bad)
    print("gex:", ct_gex[bad])
    print("rna:", ct_rna[bad])

Row-wise same cell_type: 1.0 = 74717 / 74717


In [47]:
rna_aligned_filt.write_h5ad(path + "rna_filt_aligned.h5ad", compression="gzip")
gex_aligned_filt.write_h5ad(path + "gex_filt_aligned.h5ad", compression="gzip")

# Analyse generated data

In [53]:
fake = sc.read_h5ad(path + "fake_RNA_VAE_UNET_log.h5ad")

In [58]:
fake.X.min(), fake.X.max(), fake.X.mean()

(np.float32(0.0), np.float32(12.56522), np.float32(0.033847433))

In [63]:
rna_aligned_filt.var

,feature_name
ENSG00000000003,TSPAN6
ENSG00000000005,TNMD
ENSG00000000419,DPM1
ENSG00000000457,SCYL3
ENSG00000000460,FIRRM
...,...
ENSG00000292370,NaN
ENSG00000292373,NaN
ENSG00000293546,NaN
ENSG00000293614,NaN


In [61]:
fake.var

,feature_name
ENSG00000000003,TSPAN6
ENSG00000000005,TNMD
ENSG00000000419,DPM1
ENSG00000000457,SCYL3
ENSG00000000460,FIRRM
...,...
ENSG00000292370,NaN
ENSG00000292373,NaN
ENSG00000293546,NaN
ENSG00000293614,NaN


## Analyse breast data

# Process GENESIS data

In [3]:
sriva = "/project/Wellcome_Discovery/datashare/sriva/GENESIS/"

In [4]:
# Load breast data
breast_rna = sc.read_h5ad(sriva+"RNA_filt_log_subset.h5ad")
breast_gex = sc.read_h5ad(sriva+"GEX_filt_log_subset.h5ad")

In [5]:
breast_rna.obs["cell_type"].value_counts()

cell_type
basal cell          16184
fibroblast           6526
endothelial cell     5407
T cell               2240
macrophage           1293
Name: count, dtype: int64

In [6]:
breast_gex.obs["cell_type"].value_counts()

cell_type
basal cell          16184
fibroblast           6526
endothelial cell     5407
T cell               2240
macrophage           1293
Name: count, dtype: int64

In [7]:
breast_rna.obs["cell_type"]

hbca_c157_hbca_c157_GGGTGTCAGATCGGTG-1        T cell
hbca_c157_hbca_c157_TCTGCCAGTGGCATCC-1        T cell
hbca_c157_hbca_c157_TGATGGTGTGTTCATG-1        T cell
hbca_c157_hbca_c157_TGTTGAGCATTGCTTT-1        T cell
hbca_c157_hbca_c157_TTAGGGTTCTCCCTAG-1        T cell
                                             ...    
hbca_c130_hbca_c130_CATGGATAGCTCACTA-1    macrophage
hbca_c130_hbca_c130_ATCACAGTCGACACTA-1    macrophage
hbca_c130_hbca_c130_ACTTATCGTCTACAGT-1    macrophage
hbca_c134_hbca_c134_ACACGCGTCGATCCAA-1    macrophage
r2_hbca_c88_AGCGTCGCATATCTGG-1            macrophage
Name: cell_type, Length: 31650, dtype: category
Categories (5, object): ['T cell', 'basal cell', 'endothelial cell', 'fibroblast', 'macrophage']

In [8]:
breast_gex.obs["cell_type"]

TATCACAAGGACCAGG-1_9         T cell
AGTTGGCGTTTGGGTA-1_7         T cell
CGGAGCAAGCTCAATA-1_12        T cell
CCCTGTTAGGACCTTG-1_3         T cell
CAATGTGGTAATGACT-1_9         T cell
                            ...    
CCAGCTGCACCTACGG-1_10    macrophage
CCAGGATGTTACGCAA-1_10    macrophage
CCATCACTCCTAGTCC-1_10    macrophage
GGAACAATCAATCATG-1_5     macrophage
GGCAAATCAGGTATTT-1_7     macrophage
Name: cell_type, Length: 31650, dtype: category
Categories (5, object): ['T cell', 'basal cell', 'endothelial cell', 'fibroblast', 'macrophage']